# UMAP: Uniform Manifold Approximation and Projection

## Quick Overview
**UMAP** is a **non-linear dimensionality reduction** technique that combines the best of both worlds:
- **Speed** comparable to PCA
- **Local structure preservation** like t-SNE  
- **Global structure preservation** (better than t-SNE)
- **Parametric** - can transform new data (unlike t-SNE)

## Core Idea
UMAP builds a high-dimensional graph of local neighborhoods, then layouts this graph in lower dimensions while preserving neighborhood structure. It uses topological and geometric principles to maintain both local and global patterns.

## Key Concept: Fuzzy Topological Structure
- Treats high-dimensional distances as **probabilities** (like t-SNE)
- Constructs **k-nearest neighbor graph** in high-dimensional space
- Projects to low dimensions while preserving this graph structure
- More mathematically elegant than t-SNE (based on Riemannian geometry)

## UMAP vs t-SNE

| Aspect | t-SNE | UMAP |
|--------|-------|------|
| **Speed** | 🐢 Slow | ⚡ Fast (10-100x faster) |
| **Local Structure** | ✅ Excellent | ✅ Excellent |
| **Global Structure** | ❌ Poor | ✅ Good |
| **Parametric** | ❌ No | ✅ Yes |
| **New Data** | ❌ Recompute from scratch | ✅ Transform directly |
| **Scalability** | ❌ Small datasets | ✅ Large datasets (>100k) |
| **Interpretability** | ❌ Low | ⚠️ Medium |

## Algorithm Overview

### Step 1: Build High-Dimensional Graph
- Compute k-nearest neighbors for each point
- Use fuzzy set operations to create weighted graph
- Captures local neighborhood structure

### Step 2: Initialize Low-Dimensional Layout
- Start with spectral layout (deterministic initialization)
- Much better than random initialization → faster convergence

### Step 3: Layout Optimization
- Use stochastic gradient descent to project graph
- Minimize cross-entropy between high and low-dimensional graphs
- Iteratively adjust positions

### Step 4: (Optional) Apply to New Data
- Since UMAP learns a function, can transform new unseen data
- Use fitted UMAP transformer on test set
- t-SNE cannot do this!

## Important Parameters

| Parameter | Default | Range | Meaning |
|-----------|---------|-------|---------|
| **n_neighbors** | 15 | 2-100 | Size of local neighborhood. Lower = focus on local, Higher = broader view |
| **n_components** | 2 | Any | Output dimensions (2 or 3 for visualization) |
| **metric** | 'euclidean' | 'euclidean', 'manhattan', 'cosine', etc. | Distance metric |
| **min_dist** | 0.1 | 0.0-1.0 | Minimum distance between points in output. Higher = more spread out |
| **spread** | 1.0 | 0.1-5.0 | Effective scale of embedded space. Higher = more separation |
| **learning_rate** | 1.0 | 0.01-10 | Speed of optimization |
| **n_epochs** | 200 | 10-500 | Number of optimization iterations |

### Parameter Tips
- **n_neighbors = 15**: Good default, use 5-15 for small data, 15-100 for large data
- **min_dist = 0.1**: Default balances compactness and separation
  - Lower (0.0): Points cluster tightly together
  - Higher (0.5): Points spread out more
- **metric**: Choose based on data type (euclidean for numeric, cosine for sparse/text)

## Installation & Import

```python
# Install UMAP
pip install umap-learn

# Import
import umap

# Basic usage
reducer = umap.UMAP(n_neighbors=15, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X)

# On new data
X_new_umap = reducer.transform(X_new)  # ← This is what t-SNE can't do!
```

## Practical Example 1: Basic UMAP with MNIST

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import umap
import time

# Load MNIST digits dataset
digits = load_digits()
X = digits.data  # 1797 samples, 64 features
y = digits.target  # Labels 0-9

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply UMAP
print("\nApplying UMAP...")
start_time = time.time()
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    metric='euclidean',
    random_state=42,
    verbose=1
)
X_umap = reducer.fit_transform(X_scaled)
elapsed = time.time() - start_time
print(f"Completed in {elapsed:.2f} seconds")

# Visualize
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='tab10', s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Digit')
plt.title('UMAP Visualization of MNIST Digits', fontsize=14, fontweight='bold')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✨ Observation: Clear digit clusters with both local AND global structure preserved!")

## Practical Example 2: Transform New Data (Key Advantage!)

In [ ]:
# This is what UMAP can do but t-SNE CANNOT!

# Split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Fit UMAP on training data only
print("\nFitting UMAP on training data...")
reducer_train = umap.UMAP(n_neighbors=15, n_components=2, random_state=42)
X_train_umap = reducer_train.fit_transform(X_train)

# Transform test data using the fitted model
print("Transforming test data...")
X_test_umap = reducer_train.transform(X_test)

# Visualize train and test together
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Training data
scatter1 = axes[0].scatter(X_train_umap[:, 0], X_train_umap[:, 1], c=y_train, cmap='tab10', s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
axes[0].set_title('Training Data (fitted UMAP)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
axes[0].grid(True, alpha=0.3)

# Test data (transformed using fitted model)
scatter2 = axes[1].scatter(X_test_umap[:, 0], X_test_umap[:, 1], c=y_test, cmap='tab10', s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
axes[1].set_title('Test Data (transformed with fitted model)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')
axes[1].grid(True, alpha=0.3)

plt.colorbar(scatter1, ax=axes[0], label='Digit')
plt.colorbar(scatter2, ax=axes[1], label='Digit')
plt.tight_layout()
plt.show()

print("\n🎯 Key Difference from t-SNE:")
print("✅ UMAP can transform new/test data using the fitted model")
print("❌ t-SNE must recompute from scratch for every new dataset")
print("\nThis makes UMAP suitable for ML pipelines where you need to:")
print("  • Fit on training data")
print("  • Transform test/production data consistently")

## Practical Example 3: Impact of n_neighbors Parameter

In [ ]:
# Test different n_neighbors values
n_neighbors_list = [5, 15, 50]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, n_neighbors in enumerate(n_neighbors_list):
    print(f"Computing UMAP with n_neighbors={n_neighbors}...")
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=2,
        random_state=42
    )
    X_umap_temp = reducer.fit_transform(X_scaled)
    
    scatter = axes[idx].scatter(X_umap_temp[:, 0], X_umap_temp[:, 1], 
                                c=y, cmap='tab10', s=30, alpha=0.7, edgecolors='k', linewidth=0.5)
    axes[idx].set_title(f'n_neighbors = {n_neighbors}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('UMAP 1')
    axes[idx].set_ylabel('UMAP 2')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("• Low n_neighbors (5): Focuses on very local structure")
print("• Medium n_neighbors (15): Balanced, good cluster separation")
print("• High n_neighbors (50): Emphasizes global structure")

## Advantages vs Disadvantages

### ✅ Advantages
1. **Speed**: 10-100x faster than t-SNE
2. **Scalability**: Handles datasets with 100k+ samples efficiently
3. **Parametric**: Can transform new data (great for ML pipelines)
4. **Local & Global Structure**: Better than t-SNE at preserving both
5. **Flexible**: Works with many distance metrics
6. **Deterministic Initialization**: Better reproducibility than t-SNE
7. **Beautiful Visualizations**: Creates interpretable embeddings

### ❌ Disadvantages
1. **Requires Installation**: Not built-in to scikit-learn (need `pip install umap-learn`)
2. **Hyperparameter Sensitive**: Results depend on n_neighbors, min_dist, metric
3. **Non-Linear**: Still can't capture linear relationships as well as PCA
4. **Less Intuitive**: Fewer theoretical guarantees than PCA
5. **Global Structure Preservation Trade-off**: Not as good as PCA, not as local as t-SNE

## When to Use UMAP

### ✅ Use UMAP When:
- **Need fast dimensionality reduction** for visualization
- **Dataset is large** (>10k samples) - UMAP scales well
- **Need to transform new data** (ML pipeline requirement)
- **Want both local and global structure** preserved
- **Building production systems** where consistency matters
- **Exploring high-dimensional data** (images, embeddings, etc.)
- **Need parametric transform** (can save and apply to test data)

### ❌ Don't Use UMAP When:
- **Need maximum local cluster clarity** → use t-SNE instead
- **Need linear interpretable dimensions** → use PCA
- **Speed is critical** for very first visualization → PCA is still faster
- **Working with tiny dataset** (<100 samples) → overhead not worth it
- **Need percentage variance explained** → use PCA
- **Only visualizing, never transforming new data** → t-SNE is fine

## Comparison: PCA vs t-SNE vs UMAP

| Task | Best Choice | Reason |
|------|-------------|--------|
| **Feature reduction for ML** | PCA | Fast, interpretable, parametric |
| **Beautiful visualization** | t-SNE or UMAP | Preserve local structure |
| **Large dataset viz** | UMAP | Fast, scales well |
| **Production ML pipeline** | UMAP | Parametric, consistent |
| **Explore local clusters** | t-SNE | Best local preservation |
| **Preserve global structure** | UMAP | Best of both worlds |
| **Quick EDA** | UMAP | Fast + good quality |
| **Publication-quality plot** | t-SNE | Most beautiful results |

## Best Practices & Tips

### 1. Preprocessing (Important!)
```python
from sklearn.preprocessing import StandardScaler
import umap

# Always standardize features
X_scaled = StandardScaler().fit_transform(X)

# Then apply UMAP
reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X_scaled)
```

### 2. Parameter Selection Guide
```python
# For exploration (balanced):
umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)

# For tight local clusters:
umap.UMAP(n_neighbors=5, min_dist=0.01, random_state=42)

# For broader global structure:
umap.UMAP(n_neighbors=50, min_dist=0.5, random_state=42)

# For large datasets (>100k):
umap.UMAP(n_neighbors=15, n_epochs=200, random_state=42)
```

### 3. Reproducibility
```python
# Always set random_state for consistent results
reducer = umap.UMAP(random_state=42)
```

### 4. Save & Load Models
```python
import pickle

# Save trained UMAP model
with open('umap_model.pkl', 'wb') as f:
    pickle.dump(reducer, f)

# Load and use on new data
with open('umap_model.pkl', 'rb') as f:
    loaded_reducer = pickle.load(f)
    
X_new_umap = loaded_reducer.transform(X_new)
```

### 5. Supervised UMAP (Optional)
```python
# Include label information for guided dimensionality reduction
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    random_state=42
)
# Use labels to guide embedding
X_umap = reducer.fit_transform(X_scaled, y=y)
```

### 6. 3D Visualization
```python
# For 3D plots
reducer_3d = umap.UMAP(n_components=3, random_state=42)
X_umap_3d = reducer_3d.fit_transform(X_scaled)

# Use plotly for interactive 3D
import plotly.graph_objects as go
fig = go.Figure(data=[go.Scatter3d(
    x=X_umap_3d[:, 0], y=X_umap_3d[:, 1], z=X_umap_3d[:, 2],
    mode='markers', marker=dict(color=y, colorscale='Viridis')
)])
fig.show()
```

## Common Mistakes to Avoid

### ❌ Mistake 1: Using Raw, Unscaled Data
```python
# WRONG
reducer = umap.UMAP().fit_transform(X)

# RIGHT
X_scaled = StandardScaler().fit_transform(X)
reducer = umap.UMAP().fit_transform(X_scaled)
```

### ❌ Mistake 2: Not Setting random_state
```python
# WRONG - results vary between runs
reducer = umap.UMAP()

# RIGHT - reproducible results
reducer = umap.UMAP(random_state=42)
```

### ❌ Mistake 3: Wrong n_neighbors Choice
```python
# WRONG - way too small for structure
umap.UMAP(n_neighbors=2)  # Only considers 2 neighbors

# RIGHT - balanced
umap.UMAP(n_neighbors=15)  # Standard default
```

### ❌ Mistake 4: Extreme min_dist Values
```python
# WRONG - points too compressed
umap.UMAP(min_dist=0.0)  # All points cluster tightly

# WRONG - points too sparse
umap.UMAP(min_dist=1.0)  # Loses any structure

# RIGHT - balanced
umap.UMAP(min_dist=0.1)  # Default, good spread
```

### ❌ Mistake 5: Not Using Fitted Model for Test Data
```python
# WRONG - inconsistent projections
X_train_umap = umap.UMAP().fit_transform(X_train)
X_test_umap = umap.UMAP().fit_transform(X_test)  # Different model!

# RIGHT - consistent using same model
reducer = umap.UMAP()
X_train_umap = reducer.fit_transform(X_train)
X_test_umap = reducer.transform(X_test)  # Same model, consistent space
```

## Key Takeaways 🎯

### What is UMAP?
**Non-linear dimensionality reduction** that combines speed (like PCA) with structure preservation (like t-SNE), plus parametric transformation (unique advantage).

### How Does It Work?
1. Build k-nearest neighbor graph in high-dimensional space
2. Initialize layout using spectral decomposition
3. Optimize layout in lower dimensions using fuzzy topology
4. Preserve both local and global structure

### Core Strengths
✅ **Fast** - 10-100x faster than t-SNE  
✅ **Parametric** - Transform new data consistently  
✅ **Scalable** - Handles large datasets (>100k)  
✅ **Balanced** - Preserves local AND global structure  
✅ **Beautiful** - Creates stunning visualizations  

### Core Weaknesses
❌ **Requires installation** - Not in scikit-learn by default  
❌ **Hyperparameter dependent** - Results vary with settings  
❌ **Not as local as t-SNE** - Slightly less local detail  
❌ **Not as interpretable as PCA** - No % variance explained  

### When to Choose UMAP vs Alternatives
| Situation | Choose |
|-----------|--------|
| Large dataset (>10k) + ML pipeline | ✅ **UMAP** |
| Small dataset + publication plot | t-SNE |
| Need interpretable dimensions | PCA |
| Need feature reduction only | PCA |
| Interactive exploration | ✅ **UMAP** |

### UMAP's Killer Features
1. **Parametric** - Unlike t-SNE, can transform new data
2. **Fast** - Much faster than t-SNE
3. **Balanced** - Better than t-SNE at global structure
4. **Production-ready** - Suitable for ML pipelines

### Mathematical Foundation
Based on Riemannian geometry and topological data analysis:
- Constructs manifold from high-dimensional data
- Finds low-dimensional approximation of that manifold
- Minimizes cross-entropy between fuzzy topological sets

## Further Resources

### Official Documentation
📖 **UMAP Documentation**: https://umap-learn.readthedocs.io/
📖 **GitHub Repository**: https://github.com/lmcinnes/umap

### Original Paper
📄 **"UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction"** - McInnes, Healy & Melville (2018)
- Introduces mathematical foundations
- Compares with other methods theoretically and empirically

### Recommended Articles
📚 **Why UMAP is Better Than t-SNE** - Comparison article  
📚 **Understanding UMAP** - Interactive explanations  
📚 **Topological Data Analysis** - Mathematical foundations  

### Related Techniques to Explore
- **t-SNE** - Alternative for extreme local focus
- **PCA** - Linear baseline, faster but less informative
- **Autoencoders** - Learned non-linear reduction
- **Isomap** - Geodesic distance preservation
- **DBSCAN** - Clustering that works well with UMAP output

### Implementation Libraries
```python
# UMAP (main)
from umap import UMAP

# Supervised UMAP
from umap.parametric_umap import ParametricUMAP

# Integration with scikit-learn pipeline
from sklearn.pipeline import Pipeline
```

### Quick Reference: Complete Workflow
```python
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Prepare data
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

# 2. Fit UMAP on training data
reducer = umap.UMAP(n_neighbors=15, n_components=2, random_state=42)
X_train_umap = reducer.fit_transform(X_train)

# 3. Transform test data
X_test_umap = reducer.transform(X_test)

# 4. Visualize
import matplotlib.pyplot as plt
plt.scatter(X_train_umap[:, 0], X_train_umap[:, 1], c=y_train)
plt.show()
```

---

### 🎓 Next Steps
1. **Experiment** with different n_neighbors (5, 15, 50) on your data
2. **Compare** with PCA and t-SNE to understand tradeoffs
3. **Use in pipelines** - Train UMAP on training data, transform test data
4. **Explore** supervised UMAP for label-guided embeddings
5. **Visualize** in 3D for more complex datasets
6. **Save models** and use in production systems